# LLM 실연동 테스트 노트북

Phase 4b의 Claude API 연동을 단계별로 검증.

## 사전 준비

```bash
# 1. API 키 설정 (.env 파일)
cp .env.example .env
# ANTHROPIC_API_KEY=sk-ant-xxxxx 입력

# 2. Jupyter 커널 설치 (필요 시)
uv add --dev ipykernel
uv run python -m ipykernel install --user --name portfolio-report
```

## 테스트 흐름
1. 프롬프트 미리보기 (API 호출 없음, 토큰/구조 확인)
2. 단일 종목 해석 (삼성전자) — 한 번의 Claude 호출
3. 전체 포트폴리오 파이프라인 (데이터 수집 → 지표 → LLM)
4. HTML 리포트 생성 및 브라우저 확인

In [1]:
# 프로젝트 루트를 작업 디렉토리로 설정
import os
from pathlib import Path

PROJECT_ROOT = Path('/home/shin/Project/portfolio_report')
os.chdir(PROJECT_ROOT)
print('CWD:', os.getcwd())

CWD: /home/shin/Project/portfolio_report


In [2]:
# 환경변수 로드 및 Settings 확인
import warnings
warnings.filterwarnings('ignore')

from portfolio_report.config import get_settings

settings = get_settings()
print(f'모델: {settings.claude_model}')
print(f'max_tokens: {settings.llm_max_tokens}, temperature: {settings.llm_temperature}')
print(f'API 키 설정됨: {settings.anthropic_api_key is not None}')
if settings.anthropic_api_key is None:
    print('⚠ .env에 ANTHROPIC_API_KEY를 설정해야 실제 API 호출이 가능합니다.')

모델: claude-sonnet-4-20250514
max_tokens: 1024, temperature: 0.2
API 키 설정됨: True


## Step 1 — 프롬프트 미리보기 (API 호출 없음)

실제 API에 보내기 전에, `prompts.py`가 만들어내는 텍스트를 눈으로 확인.

In [3]:
from portfolio_report.llm.base import TechnicalContext
from portfolio_report.llm.prompts import (
    build_technical_system_prompt,
    build_technical_user_prompt,
)

sample_ctx = TechnicalContext(
    code='005930',
    name='삼성전자',
    current_price=67_000.0,
    signals={
        'ichimoku': {
            'close': 67000, 'tenkan': 66500, 'kijun': 65500,
            'span_a': 64000, 'span_b': 62500,
            'signal': '구름대 위 (강세), 전환선>기준선 (단기 강세)',
        },
        'rsi': {'value': 72.5, 'signal': '과매수 (72.5)'},
        'macd': {
            'macd': 450.0, 'signal_line': 380.0, 'histogram': 70.0,
            'signal': '강세',
        },
        'bb': {
            'upper': 68500, 'middle': 65000, 'lower': 61500,
            'percent_b': 0.93,
            'signal': '상단 근접',
        },
    },
    price_tail=[
        {'Date': '2026-04-11', 'Open': 65500, 'High': 66300, 'Low': 65200, 'Close': 66000, 'Volume': 12_000_000},
        {'Date': '2026-04-12', 'Open': 66000, 'High': 67200, 'Low': 65800, 'Close': 67000, 'Volume': 15_000_000},
    ],
)

print('--- SYSTEM ---')
print(build_technical_system_prompt())
print()
print('--- USER ---')
print(build_technical_user_prompt(sample_ctx))

--- SYSTEM ---
당신은 한국 주식시장의 기술적 분석 애널리스트입니다. 사용자가 제공한 수치만을 근거로 간결하고 정확한 해석을 한국어로 제공합니다. 제공되지 않은 지표·뉴스·외부 사실을 추정하거나 언급하지 않으며, 명시적으로 제공된 신호 및 수치를 중심으로 설명합니다. 본 해석은 참고용이며 투자 자문이 아님을 마지막에 한 줄로 덧붙입니다.

--- USER ---
## 종목: 삼성전자 (005930)
현재가: 67,000원

## 기술적 지표 신호
### 일목균형표
- close: 67,000
- tenkan: 66,500
- kijun: 65,500
- span_a: 64,000
- span_b: 62,500
- **신호: 구름대 위 (강세), 전환선>기준선 (단기 강세)**

### RSI (14일)
- value: 72.50
- **신호: 과매수 (72.5)**

### MACD (12/26/9)
- macd: 450.00
- signal_line: 380.00
- histogram: 70.00
- **신호: 강세**

### 볼린저밴드 (20일 ±2σ)
- upper: 68,500
- middle: 65,000
- lower: 61,500
- percent_b: 0.93
- **신호: 상단 근접**

## 최근 시세 (최근 N일)
| 날짜 | 시가 | 고가 | 저가 | 종가 | 거래량 |
|---|---|---|---|---|---|
| 2026-04-11 | 65,500 | 66,300 | 65,200 | 66,000 | 12,000,000 |
| 2026-04-12 | 66,000 | 67,200 | 65,800 | 67,000 | 15,000,000 |

## 요청
위 수치만을 근거로 이 종목의 최근 기술적 흐름을 3~5문장으로 해석해 주세요. 과매수/과매도, 추세, 주요 지지·저항 수준의 시사점을 포함하되, 제공되지 않은 값(예: 목표주가, 뉴스, 재무지표)은 언급하지 않습니다.


## Step 2 — 단일 종목 해석 (Claude API 1회 호출)

- 예상 비용: claude-sonnet-4 기준 약 0.003~0.01 USD
- 실패 시 `(LLM 해석 실패: ...)` 문자열 반환

In [4]:
from portfolio_report.llm.claude_client import ClaudeClient

client = ClaudeClient()
explanation = client.explain_technical(sample_ctx)
print(explanation)

삼성전자는 현재 일목균형표 구름대(62,500~64,000원) 위에서 거래되고 있으며, 전환선(66,500원)이 기준선(65,500원)을 상회하여 단기 강세 신호를 보이고 있습니다. MACD는 450.00으로 신호선(380.00) 대비 상당한 상승 모멘텀을 나타내고 있으나, RSI가 72.5로 과매수 구간에 진입한 상태입니다. 

볼린저밴드에서 현재가가 상단(68,500원)에 근접(퍼센트B 0.93)해 있어 단기 조정 가능성을 시사하며, 중심선인 65,000원이 주요 지지선으로 작용할 것으로 보입니다. 최근 이틀간 거래량 증가와 함께 상승세를 보였으나, 과매수 신호와 볼린저밴드 상단 근접으로 인해 단기적으로는 신중한 접근이 필요한 상황입니다.

※ 본 해석은 기술적 분석에 따른 참고용 정보이며 투자 자문이 아닙니다.


## Step 3 — 실데이터 기반 단일 종목 해석

픽스처 대신 실제 FDR OHLCV + 네이버 PER/베타를 사용.

In [5]:
from portfolio_report.data.naver_client import NaverClient
from portfolio_report.data.price_client import PriceClient
from portfolio_report.analysis.technical import compute_indicators

code = '005930'
with NaverClient() as nc:
    snap = nc.fetch_snapshot(code)

pc = PriceClient()
ohlcv = pc.get_ohlcv(code, days=180)
result = compute_indicators(ohlcv, ['ichimoku', 'rsi', 'macd', 'bb'])

print('현재가:', snap.get('current_price'))
print('신호:', result.signals)
print('OHLCV 행 수:', len(result.df))

현재가: 211000.0
신호: {'ichimoku': {'close': 211000.0, 'tenkan': 199100.0, 'kijun': 191250.0, 'span_a': 190925.0, 'span_b': 164600.0, 'signal': '구름대 위 (강세), 전환선>기준선 (단기 강세)'}, 'rsi': {'value': 59.991341242755695, 'signal': '중립 (60.0)'}, 'macd': {'macd': 6193.938750767236, 'signal_line': 4257.626053243139, 'histogram': 1936.3126975240975, 'signal': '강세'}, 'bb': {'upper': 217269.43620545964, 'middle': 192550.0, 'lower': 167830.56379454036, 'percent_b': 0.8731881230350442, 'signal': '상단 근접'}}
OHLCV 행 수: 139


In [6]:
# 실데이터 컨텍스트로 Claude 호출
real_tail = [
    {**row, 'Date': idx.strftime('%Y-%m-%d')}
    for idx, row in ohlcv.tail(10).to_dict(orient='index').items()
]

real_ctx = TechnicalContext(
    code=code,
    name=snap.get('name', '삼성전자'),
    current_price=snap.get('current_price'),
    signals=result.signals,
    price_tail=real_tail,
)

print(client.explain_technical(real_ctx))

삼성전자는 현재 일목균형표 구름대(164,600~190,925) 위에서 거래되며 전환선(199,100)이 기준선(191,250)을 상회하여 단기 강세 신호를 보이고 있습니다. MACD는 6,194로 신호선(4,258)을 상회하며 히스토그램이 양수(1,936)를 기록해 상승 모멘텀이 지속되고 있음을 시사합니다. 

볼린저밴드에서 %B가 0.87로 상단밴드(217,269) 근접 수준이며, RSI는 59.99로 중립권 상단에 위치해 과열 우려는 제한적입니다. 최근 시세를 보면 4월 2일 178,400원 저점 이후 지속적인 상승세를 보이며 현재 211,000원까지 회복했으나, 볼린저밴드 상단이 단기 저항으로 작용할 가능성이 있습니다.

※ 본 해석은 기술적 분석에 기반한 참고용 정보이며 투자 자문이 아닙니다.


## Step 4 — 전체 포트폴리오 파이프라인

`PortfolioAnalyzer.analyze(..., use_llm=True)`로 모든 종목을 한 번에 해석.

In [7]:
from portfolio_report.models.holding import HoldingInput
from portfolio_report.analysis.aggregator import PortfolioAnalyzer

inputs = [
    HoldingInput(name='삼성전자', quantity=10),
    HoldingInput(name='SK하이닉스', quantity=5),
    HoldingInput(code='035420', quantity=3),
]

analyzer = PortfolioAnalyzer()
report = analyzer.analyze(inputs, indicators=['ichimoku', 'rsi', 'macd', 'bb'], use_llm=True)

print(f'총 평가금액: {report.aggregates.total_market_value:,.0f}원')
print(f'가중 PER: {report.aggregates.weighted_per:.2f}')
print(f'가중 베타: {report.aggregates.weighted_beta:.2f}')
print()
print('LLM 해석이 붙은 종목 수:', sum(1 for a in report.per_stock_analyses if a.llm_explanation))

보통주 '삼성전자'으로 해석했습니다. 우선주도 상장되어 있습니다: ['삼성전자우']. 우선주를 의도한 경우 종목코드를 직접 지정하세요.


총 평가금액: 8,423,000원
가중 PER: 22.33
가중 베타: 1.44

LLM 해석이 붙은 종목 수: 3


In [8]:
# 각 종목의 LLM 해석 출력
for analysis in report.per_stock_analyses:
    print('=' * 70)
    print(f'{analysis.code} {analysis.name}')
    print('-' * 70)
    print('신호 요약:')
    for ind, data in analysis.indicators.items():
        print(f'  {ind}: {data.get("signal", "?")}')
    print()
    print('해석:')
    print(analysis.llm_explanation or '(없음)')
    print()

005930 삼성전자
----------------------------------------------------------------------
신호 요약:
  ichimoku: 구름대 위 (강세), 전환선>기준선 (단기 강세)
  rsi: 중립 (60.0)
  macd: 강세
  bb: 상단 근접

해석:
## 삼성전자 기술적 분석 해석

삼성전자는 현재 일목균형표 구름대(164,600~190,925원) 위에서 거래되며 전환선(199,100원)이 기준선(191,250원)을 상회하여 단기 강세 신호를 보이고 있습니다. MACD는 6,194로 신호선(4,258)을 상회하며 히스토그램(1,936)이 양수를 유지해 상승 모멘텀이 지속되고 있음을 시사합니다. 

볼린저밴드에서 %B값이 0.87로 상단밴드(217,269원) 근처에 위치하여 단기 과열 가능성을 보여주며, RSI는 59.99로 중립권에서 60선 근접으로 추가 상승 여력이 제한적일 수 있습니다. 최근 시세를 보면 4월 초 175,000원 저점에서 현재 211,000원까지 상승하며 강한 회복세를 보였으나, 볼린저밴드 상단 근처에서 단기 조정 압력이 존재할 것으로 판단됩니다.

*본 해석은 기술적 분석에 기반한 참고용 정보이며 투자 자문이 아닙니다.

000660 SK하이닉스
----------------------------------------------------------------------
신호 요약:
  ichimoku: 구름대 위 (강세), 전환선>기준선 (단기 강세)
  rsi: 중립 (64.6)
  macd: 강세
  bb: 상단 돌파 (과열)

해석:
SK하이닉스는 최근 10거래일 동안 830,000원에서 1,136,000원까지 약 37% 급등하며 강한 상승 추세를 보이고 있습니다. 일목균형표에서 현재가가 구름대(823,000~949,250원) 위에 위치하고 전환선(1,016,500원)이 기준선(989,500원)을 상회하여 단기 강세 신호를 나타내며, MACD도 39,689으로 시그널라인

## Step 5 — HTML 리포트 저장 후 브라우저에서 확인

In [9]:
from datetime import datetime
from portfolio_report.reporting.html import render_html

html = render_html(report)
out = Path('data/reports') / f'llm_test_{datetime.now():%Y%m%d_%H%M%S}.html'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(html, encoding='utf-8')
print(f'✓ 저장됨: {out}')
print(f'  크기: {len(html):,} bytes')
print(f'  브라우저에서 열기: file://{out.resolve()}')

✓ 저장됨: data/reports/llm_test_20260415_160041.html
  크기: 219,431 bytes
  브라우저에서 열기: file:///home/shin/Project/portfolio_report/data/reports/llm_test_20260415_160041.html


## 점검 포인트

- [ ] 해석 텍스트가 한국어로 나오는가
- [ ] 제공된 수치(RSI 값, 구름대 위치 등)를 인용하는가
- [ ] 제공되지 않은 정보(목표주가, 뉴스)를 언급하지 않는가
- [ ] 마지막에 '참고용' 면책 문구가 포함되는가
- [ ] API 비용이 예상 범위 내인가 (3종목 × 1호출 ≈ 0.01~0.03 USD)